# analysis.detection-statistics


## Summary 

In this notebook, we focus on the numbers generated by our bacteria-taxonomic profiling using MOTUs. We read mostly content from the `metadata.site-library.csv` files and from `hits.bacteria.csv`. We first carry out a bacteria / PAB based analysis, and then we carry out a library based analysis. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import taxoniq
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

## Bacteria

We carry out the analysis here by making questions and answering them.

### How many bacteria were detected?

To know that, first we need to load the data. 

In [ ]:
bacteria_hits = db.conn.sql('SELECT * FROM D_bacteriaHits').df()
metadata = db.conn.sql('SELECT * FROM D_sites').df()

Now, we value count the different taxids. 

In [ ]:
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left')#.dropna(subset='taxid')
# bacteria_hits['taxid'] = bacteria_hits['taxid'].astype(int)
bacteria_hits

The data contains hits. We need to aggregate the hits to get the OTUs to which they map, which we do by a simple `value_counts`. 

In [ ]:
bacteria_hits_count = bacteria_hits.value_counts(['scientific_name']).reset_index()
bacteria_hits_count['rank'] = 1 + np.arange(len(bacteria_hits_count))
bacteria_hits_count['%'] = bacteria_hits_count['count'] / len(bacteria_hits)
bacteria_hits_count

In [ ]:
bacteria_hits_count.query('scientific_name == ""')

In [ ]:
print("Total hits {:6d}".format(len(bacteria_hits))) # 1517 q000008
print(" |--> corresponding to {:6d} bacterial OTUs".format(len(bacteria_hits_count))) # 520 q000010


We have 1486 hits corresponding to 519 OTUs. Now, let's see how they distribute. 

In [ ]:
pd.DataFrame.from_records(
    [
        {"threshold": "=1", "count": len(bacteria_hits_count.query('count == 1'))}, # 311
        {"threshold": ">1", "count": len(bacteria_hits_count.query('count > 1'))}, # 211
        {"threshold": ">2", "count": len(bacteria_hits_count.query('count > 2'))}, # 113 
        {"threshold": ">5", "count": len(bacteria_hits_count.query('count > 5'))}, # 43
        {"threshold": ">10", "count": len(bacteria_hits_count.query('count > 10'))}, # 22
    ]
)



Most OTUs (> 300) have been detected only one. A ranking map of these OTUs should help us see this pattern. In the folllowing plot, we are sorting by growing rank the OTUs depending on their number of hits. On the left, we would find the OTUs with higher number of hits, on the right those with the lower number of hits. 

In [ ]:
g = sns.relplot(
    data=bacteria_hits_count, y='count', x='rank', kind='line',
    height=2.0, aspect=2.5
)
# g.ax.axvline(10, linestyle='--', color='gray')
g.ax.set_yscale('log')


We can observe indeed a very heterogeneous distribution, with some hits exceeding the 100 detections. Let's see which could be these organisms.

In [ ]:
g = sns.catplot(
    data=bacteria_hits_count[:20], y='scientific_name', x='%', kind='bar', height=4.0, aspect=1.0
)
g.ax.set_xscale('log')

This is quite interesting:
- The most abundant organism is an *uncultured Clostridium sp.*. This means that it could be whatever.
- After that one, we have some organism that are indeed known for interacting with plants such as *Pseudomonas lutea*.
- We also find unexpected items, such as *Escherichia coli* and *Mycoplasmoides genitalium*. This could mean that our samples are either contaminated by animal associated bacteria.

#### Conclussion

- 1517 hits
- 520 OTUs
- Very heterogeneous distribution
- There are some unexpected organisms

### How many bacteria are plant-associated-bacteria?

The way to solve this query is very easy: just remove the bacteria that are not PAB, and repeat the whole analysis. 

In [ ]:
bacteria_hits_count = bacteria_hits.query('is_pab == True').value_counts(['scientific_name', 'pab_type']).reset_index()
bacteria_hits_count['rank'] = 1 + np.arange(len(bacteria_hits_count))
bacteria_hits_count['%'] = bacteria_hits_count['count'] / len(bacteria_hits.query('is_pab == True'))
bacteria_hits_count

In [ ]:
print("Total PAB hits {:6d}".format(len(bacteria_hits.query('is_pab == True')))) # 433 q000011
print(" |--> corresponding to {:6d} PAB OTUs".format(len(bacteria_hits_count))) # 127 q000012


We have a total of 433 PAB hits, corresponding to 127 OTUs. These numbers might have been slightly higher in the past, given some errors when counting organisms. Let's take again one look at the count.

In [ ]:
pd.DataFrame.from_records(
    [
        {"threshold": "=1", "count": len(bacteria_hits_count.query('count == 1'))},# 53 
        {"threshold": ">1", "count": len(bacteria_hits_count.query('count > 1'))}, # 74
        {"threshold": ">2", "count": len(bacteria_hits_count.query('count > 2'))}, #45
        {"threshold": ">5", "count": len(bacteria_hits_count.query('count > 5'))}, # 20
        {"threshold": ">10", "count": len(bacteria_hits_count.query('count > 10'))}, # 9
    ]
)

The numbers are quite brutal again. Two fifths of the data have only one positive hit. Let's look at this again with a rank-plot. 

In [ ]:
g = sns.relplot(
    data=bacteria_hits_count, y='count', x='rank', kind='line',
    height=2.0, aspect=2.0
)
g.ax.set_yscale('log')
g.set_xlabels("OTU rank")
g.set_ylabels("# hits")

g.savefig("figures/rankplot.pab-bacteria.svg")

Now we can take a quick look at the composition of these bacteria.

In [ ]:
g = sns.catplot(
    data=bacteria_hits_count[:20], y='scientific_name', x='%', kind='bar'
)
# g.ax.set_xscale('log')

### What is the high-level taxonomic composition of our PABs?

While we have the species represented in the plot above, higher-taxonomy levels might provide another picture. We will use Taxoniq to compute the taxon, class, order, etc of each of the taxa, and we will dump it in a figure. 

In [ ]:
bacteria_hits_count['taxon'] = bacteria_hits_count['taxid'].apply(lambda x: taxoniq.Taxon(x))
bacteria_hits_count['class'] = bacteria_hits_count['taxon'].apply(lambda x: [item.scientific_name for item in x.ranked_lineage if item.rank.name == 'class'][0])
bacteria_hits_count['order'] = bacteria_hits_count['taxon'].apply(lambda x: [item.scientific_name for item in x.ranked_lineage if item.rank.name == 'order'][0])
bacteria_hits_count['phylum'] = bacteria_hits_count['taxon'].apply(lambda x: [item.scientific_name for item in x.ranked_lineage if item.rank.name == 'phylum'][0])

The following figure should represent the phylum (color) and the order (row) of all the species detected. 

In [ ]:
bacteria_taxonomies = bacteria_hits_count.value_counts(['order', 'phylum']).reset_index()
top10 = bacteria_taxonomies.groupby(['order'])['count'].sum().reset_index().sort_values(by='count', ascending=False)[:10].order.values
bacteria_taxonomies['order'] = bacteria_taxonomies['order'].apply(lambda x: x if x in top10 else 'Other')
bacteria_taxonomies['phylum'] = bacteria_taxonomies.apply(lambda x: x.phylum if x.order != 'Other' else 'Other', axis=1)
db.save_dataframe(
    bacteria_taxonomies, table_name="P_PABtaxonomy",
    description="Number of PAB hits belonging to each different phylum and order"
)
g = sns.catplot(
    bacteria_taxonomies, x='count', hue='phylum', y='order', kind='bar', edgecolor='black', height=2., aspect=2.5, dodge=False,
    palette='Set2'
)
g.set_xlabels("OTUs")
g.savefig("figures/catplot.detections-taxonomic-composition.colbyphyla.svg")

### What kind of PABs?

Sanchis et al. (2021) consider three types of PABs:
- *symbionts*, which we prefer to call *mutualists*
- *pathogens*, which we should call *antagonists* —but we won't.
- *unknown*, those that have an uncertain relationship with the hosts.

We are going to generate a table the considers the numbers of each of these PAB types among our OTUs.

In [ ]:
pab_types = bacteria_hits_count.value_counts(['pab_type']).reset_index().rename(columns={'count': 'OTUs'})
db.save_dataframe(
    pab_types, table_name="D_PABTypeCounts",
    description="Occurrences of the different kinds of PABs"
)
pab_types

## Libraries

Again, simple Q & A

### What is the distribution of hits across our libraries?

To answer this question, first we need to value-count the number of total hits per library. We all save the habitat. 

In [ ]:
library_hits_count = bacteria_hits.value_counts(['library', 'habitat']).reset_index()
library_hits_count['rank'] = np.arange(len(library_hits_count))
library_hits_count

The following table indicates some values of the distribution.

In [ ]:
pd.DataFrame.from_records(
    [
        {"threshold": "=1", "count": len(library_hits_count.query('count > 0'))}, #323
        {"threshold": "=1", "count": len(library_hits_count.query('count == 1'))}, #107
        {"threshold": ">1", "count": len(library_hits_count.query('count > 1'))}, #211
        {"threshold": ">2", "count": len(library_hits_count.query('count > 2'))},
        {"threshold": ">5", "count": len(library_hits_count.query('count > 5'))},
        {"threshold": ">10", "count": len(library_hits_count.query('count > 10'))},
    ]
)



We have 299 positive libraries out of the 323 libraries, which means 24 negatives. Only 22 libraries have more than 10 OTUs detected.

We use the following rank plot to visualize the heterogeneity of host ranges.

In [ ]:
res_ = library_hits_count.sort_values(by='count', ascending=False).query('count > 0')
res_['rank'] = np.arange(1, len(res_) + 1)

# db.save_dataframe(
#     res_, table_name="site_rank_plot",
#     description="Ranking of sites depending on their detections"
# )

g = sns.relplot(
    data=res_,
    x='rank', y='count', kind='line',
    height=2.0, aspect=2.0
)
g.ax.set_yscale('log')
g.set_xlabels("rank")
g.set_ylabels("# hits per library")

### How many PAB detections per library?

As simple as the analysis above, but just analyzing PAB hits.

In [ ]:
library_hits_count = bacteria_hits.query('is_pab == True').value_counts(['library', 'site', 'habitat']).reset_index()
library_hits_count['rank'] = np.arange(len(library_hits_count))
library_hits_count

In [ ]:
pd.DataFrame.from_records(
    [
        {"threshold": "=1", "count": len(library_hits_count.query('count == 1'))},
        {"threshold": ">1", "count": len(library_hits_count.query('count > 1'))},
        {"threshold": ">2", "count": len(library_hits_count.query('count > 2'))},
        {"threshold": ">5", "count": len(library_hits_count.query('count > 5'))},
        {"threshold": ">10", "count": len(library_hits_count.query('count > 10'))},
    ]
)



Only 136 libraries were positive to PABs.

In [ ]:
res_ = library_hits_count.sort_values(by='count', ascending=False).query('count > 0')
res_['rank'] = np.arange(1, len(res_) + 1)

# db.save_dataframe(
#     res_, table_name="site_rank_plot_PAB",
#     description="Ranking of sites depending on their detections of PAB OTUs"
# )

g = sns.relplot(
    data=res_,
    x='rank', y='count', kind='line',
    height=2.0, aspect=2.0
)
g.ax.set_yscale('log')
g.set_xlabels("Library rank")
g.set_ylabels("# hits")
g.savefig("figures/rankplot.pab-library.svg")

## Conclussions

1. We are able to detect 1486 bacteria hits, corresponding to 522 OTUs.
2. From those, around one third of the hits correspond to plant-associated bacteria. The total PAB OTUs are 127.
3. Most of the hits correspond to the order hyphomicrobiales, microccocales, burkhlodeliares, and pseudomonareales #TODO: check spelling.
4. Only 136 libraries (around half of them) were positive to PABs.

In [ ]:
db.conn.close()